# SIS6116 — MAX–MIN Ant System on a personalised symmetric TSP

This notebook runs one fixed MAX–MIN Ant System (MMAS) baseline on the seed-4616 travelling salesperson instance and writes the results to `outputs/`. It does not retune parameters.

The search is `sis6116.experiments.run_baseline_experiment`. That function builds the instance, evaluates both nearest-neighbour references, and runs MMAS once for each optimiser seed. The figures and the summary table shown below are produced by that run.


## Environment

The next cell makes `sis6116` importable. In Google Colab it clones this repository into `/content` and installs `requirements.txt`. Locally, it uses the checkout that contains the notebook, including a kernel started in `notebooks/`. No packages are added beyond that requirements file.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab has no checkout, so the branch below clones one. A local kernel uses this repository.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/gopalenparasuraman/graph-problem-swarm-intelligence.git"
CLONE_ROOT = Path("/content/graph-problem-swarm-intelligence")

if IN_COLAB:
    # A shallow clone is enough to import sis6116. Skip it if /content already has the repo.
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_ROOT)])
    ROOT = CLONE_ROOT
else:
    # Opening the notebook from notebooks/ puts the package one directory above the kernel.
    cwd = Path.cwd().resolve()
    ROOT = cwd if (cwd / "sis6116").is_dir() else cwd.parent
    if not (ROOT / "sis6116").is_dir():
        raise FileNotFoundError("Run this notebook from the repository or its notebooks directory.")

# Imports below must resolve the package from the repository root.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Same dependency file as a local install. This cell adds no further packages.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")]
)
print("root:", ROOT)


## Baseline run

The next cell calls the fixed baseline. Before the search it prints the variant mechanics and the settings. Afterwards it prints the summary computed from the finished runs.

### Personalised instance

Seed 4616 fixes the graph. The city count is $N = 40 + (4616 \bmod 11) = 47$. One NumPy generator, `default_rng(4616)`, draws an $N \times N$ matrix with integer entries from 1 to 99. The distance matrix is the floored average of that matrix and its transpose, and the diagonal is set to 0. Edge weights are therefore integers, and $d_{ij} = d_{ji}$. The construction does not place the cities in the plane and does not enforce the triangle inequality, so the instance is not assumed to be Euclidean or metric.

### Tour and objective

A feasible tour is a permutation of the cities $0, \ldots, N-1$. Its cost is the sum of the $N$ cycle edges. The sum includes the return from the last city to the first. MMAS minimises that cost. The best cost returned by the budget below is the best cost observed. It is not a proof of global optimality.

### Nearest neighbour

From the current city, nearest neighbour steps to the unvisited city of least distance. A tie keeps the lowest city index. Two references use the same matrix. The tour from city 0 is also the incumbent MMAS starts from, stored as iteration 0. Best-of-all-starts repeats the rule from every city and keeps the shortest tour, breaking equal costs toward the lowest start index. City 0 is only one of the 47 starts, so the best-start tour is the stronger deterministic reference.

### How this MMAS run searches

Pheromone $\tau_{ij}$ is the learned weight of an edge. Heuristic information is $\eta_{ij} = (1/d_{ij})^{\beta}$, which favours a short edge when $\beta > 0$. The baseline sets $\alpha = 1$, $\beta = 2$ and evaporation $\rho = 0.20$. $\alpha$ weights pheromone, $\beta$ weights distance, and $\rho$ is the fraction removed from every trail before the deposit.

Each ant draws a start city uniformly, then chooses the next city only from the cities it has not visited:

$$
P(i,j) \propto \tau_{ij}^{\alpha}\,\eta_{ij}.
$$

Every ant in an iteration sees the same frozen pheromone. The iteration-best tour is the lowest cost, and equal costs keep the lowest ant index. Only that tour deposits $1/L$ on both directions of each of its edges, including the return. The global best changes only when a tour is strictly shorter. A new global best, still before evaporation, sets

$$
\tau_{\max} = \frac{1}{\rho L_{\mathrm{best}}}, \qquad \tau_{\min} = \frac{\tau_{\max}}{2N}.
$$

Trails are then multiplied by $1-\rho$, the iteration-best deposit is added, and off-diagonal entries are clipped to $[\tau_{\min}, \tau_{\max}]$. The diagonal stays 0. The ratio $\tau_{\max}/\tau_{\min} = 2N = 94$ is a simplified MMAS bound, not the $p_{\mathrm{best}}$ rule. This baseline does not run 2-opt, a stagnation restart, or an Ant Colony System local update.

### Budget held fixed

$\rho = 0.20$ is the only evaporation rate used here. Ant count stays at $m = N = 47$ and the iteration limit stays at 100, so each run constructs $47 \times 100$ tours. Seeds 0 through 9 give ten runs. Each run draws start cities and transitions from its own `default_rng(optimiser_seed)`. The instance generator is a separate stream.

A separate program, `sis6116.rho_experiment`, compares three evaporation rates with those other factors unchanged: C0 at $\rho = 0.20$ (this baseline), C1 at $\rho = 0.05$, and C2 at $\rho = 0.50$. This notebook does not call that program. It therefore does not draw the three-configuration convergence chart, the improvement-timing chart, or the pheromone-floor chart. Each MMAS run does compute bound-fraction diagnostics after the pheromone update, but this baseline neither saves nor plots them.

### What the two figures and the table show

The first figure is the mean best-so-far cost from iteration 0 to 100. The shaded band is that mean $\pm 1$ sample standard deviation across the ten runs, with divisor $n-1$. It is not a confidence interval. Dashed and dotted lines mark the city-0 and best-start nearest-neighbour costs. The second figure draws one best-so-far curve for each optimiser seed.

The printed table gives, for MMAS, the mean of the ten final costs, the sample standard deviation of those costs, the mean iteration of the last strict improvement, and the mean runtime. Runtime is the wall-clock time inside one MMAS run, including its nearest-neighbour initialisation and all 100 iterations. It excludes instance generation, the separate nearest-neighbour reference calls, plotting, and file output. Nearest neighbour is one deterministic call, so its standard-deviation and last-improvement cells are left blank. The lowest of the ten MMAS final costs is the best tour observed in this baseline, not a proved minimum.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from sis6116.experiments import run_baseline_experiment

# Fixed baseline on seed 4616. Writes CSV, JSON, and the two convergence figures under outputs/.
result = run_baseline_experiment(Path("outputs"))
# Mean best-so-far curve. The band is ±1 sample SD across the ten runs, not a confidence interval.
display(Image(filename=str(result.paths["convergence_mean_std"])))
# The same histories drawn separately, one curve per optimiser seed.
display(Image(filename=str(result.paths["convergence_overlay"])))
# Mean final cost, sample SD, mean last-improvement iteration, and mean runtime.
print(result.summary_text)
print("wrote", result.output_dir)
